<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_06_1_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# T81-558: Anwendungen tiefer neuronaler Netzwerke
**Modul 6: ChatGPT und große Sprachmodelle**
* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Modul 6 Material

* **Teil 6.1: Einführung in Transformatoren** [[Video]](https://www.youtube.com/watch?v=mn6r5PYJcu0&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=mn6r5PYJcu0&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 6.2: Zugriff auf die ChatGPT-API [[Video]](https://www.youtube.com/watch?v=tcdscXl4o5w&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=tcdscXl4o5w&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 6.3: LLM-Speicher [[Video]](https://www.youtube.com/watch?v=oGQ3TQx1Qs8&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=oGQ3TQx1Qs8&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 6.4: Einführung in Embeddings [[Video]](https://www.youtube.com/watch?v=e6kcs9Uj_ps&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=e6kcs9Uj_ps&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 6.5: Prompt Engineering [[Video]](https://www.youtube.com/watch?v=miTpIDR7k6c&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=miTpIDR7k6c&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)

# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab die richtige Version von TensorFlow ausführt.
Durch Ausführen des folgenden Codes wird Ihr GDrive ```/content/drive``` zugeordnet.

In [ ]:
try:
    from google.colab import drive

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

Note: not using Google CoLab


# Teil 6.1: Einführung in Transformatoren

Transformatoren sind neuronale Netzwerke, die moderne Lösungen für viele Probleme bieten, die früher rekurrierenden neuronalen Netzwerken vorbehalten waren. [[Cite:vaswani2017attention]](https://arxiv.org/abs/1706.03762) Sequenzen können sowohl die Eingabe als auch die Ausgabe eines neuronalen Netzwerks bilden. Beispiele für solche Konfigurationen sind:

* Vektor zur Sequenz - Bildbeschriftung
* Sequenz zu Vektor - Stimmungsanalyse
* Sequenz zu Sequenz - Sprachübersetzung

Sequenz-zu-Sequenz ermöglicht es einer Eingabesequenz, eine Ausgabesequenz basierend auf einer Eingabesequenz zu erzeugen. Transformer konzentrieren sich hauptsächlich auf diese Sequenz-zu-Sequenz-Konfiguration.

## Allgemeiner Überblick über Transformatoren

Dieser Kurs konzentriert sich hauptsächlich auf die Anwendung tiefer neuronaler Netzwerke. Der Schwerpunkt liegt auf der Präsentation von Daten für einen Transformator und den Hauptkomponenten eines Transformators. Daher konzentrieren wir uns nicht auf die Implementierung eines Transformators auf der untersten Ebene. Der folgende Abschnitt bietet einen Überblick über kritische interne Teile eines Transformators, wie z. B. Restverbindungen und Aufmerksamkeit. Im nächsten Kapitel verwenden wir Transformatoren von [Hugging Face](https://huggingface.co/), um natürliche Sprachverarbeitung mit Transformatoren durchzuführen. Wenn Sie daran interessiert sind, einen Transformator von Grund auf zu implementieren, bietet Keras einen umfassenden [Hugging Face](https://huggingface.co/).

Abbildung 10.TRANS-1 zeigt eine Übersicht über einen Transformator zur Sprachübersetzung.

**Abbildung 10.TRANS-1: Übersicht über einen Übersetzungstransformator**
![Transformer](https://data.heatonresearch.com/images/jupyter/transformer-1.jpg)

Für dieses Beispiel verwenden wir einen Transformer, der zwischen Englisch und Spanisch übersetzt. Wir geben den englischen Satz „die Katze mag Milch“ ein und erhalten die spanische Übersetzung „al gato le gusta la leche“.

Wir beginnen damit, den englischen Quellsatz zwischen den Anfangs- und Endtoken zu platzieren. Diese Eingabe kann beliebig lang sein und wir haben sie dem neuronalen Netzwerk als unregelmäßigen Tensor präsentiert. Da der Tensor unregelmäßig ist, ist keine Auffüllung erforderlich. Eine solche Eingabe ist für die Aufmerksamkeitsebene akzeptabel, die den Quellsatz empfängt. Der coder wandelt diese unregelmäßige Eingabe in einen verborgenen Zustand um, der eine Reihe von Schlüssel-Wert-Paaren enthält, die das Wissen im Quellsatz darstellen. Der coder kann Englisch lesen und in einen verborgenen Zustand umwandeln. Der Decoder kann aus diesem verborgenen Zustand Spanisch ausgeben.

Wir präsentieren dem Decoder zunächst den verborgenen Zustand und das Starttoken. Der Decoder sagt die Wahrscheinlichkeiten aller Wörter in seinem Vokabular voraus. Das Wort mit der höchsten Wahrscheinlichkeit ist das erste Wort des Satzes.

Das Wort mit der höchsten Wahrscheinlichkeit wird an den übersetzten Satz angehängt, der zunächst nur das Anfangstoken enthält. Dieser Prozess wird fortgesetzt und der übersetzte Satz wird mit jeder Iteration erweitert, bis der Decoder das Endtoken vorhersagt.

## Transformer-Hyperparameter

Bevor wir beschreiben, wie diese Schichten zusammenpassen, müssen wir die folgenden Transformator-Hyperparameter sowie die Standardeinstellungen aus dem Keras-Transformator-Beispiel berücksichtigen:

* Anzahl_Ebenen = 4
* d_Modell = 128
* dff = 512
* Anzahl Köpfe = 8
* Abbruchrate = 0,1

Es können mehrere coder- und Decoder-Ebenen vorhanden sein. Der Hyperparameter **num_layers** gibt an, wie viele coder- und Decoder-Ebenen vorhanden sind. Die erwartete Tensorform für die Eingabe in die coder-Ebene ist dieselbe wie die erzeugte Ausgabe. Daher können Sie diese Ebenen problemlos stapeln.

Im nächsten Kapitel werden wir uns Einbettungsebenen ansehen. Sie können sich eine Einbettungsebene jedoch zunächst als Wörterbuch vorstellen. Jeder Eintrag in der Einbettung entspricht jedem Wort in einem Vokabular mit fester Größe. Ähnliche Wörter sollten ähnliche Vektoren haben. Der Hyperparameter **d_model** gibt die Größe des Einbettungsvektors an. Obwohl Sie manchmal Einbettungen aus einem Projekt wie [Word2vec](https://radimrehurek.com/gensim/models/word2vec.html) oder [Word2vec](https://radimrehurek.com/gensim/models/word2vec.html) vorladen, kann der Optimierer diese Einbettungen mit dem Rest des Transformators trainieren. Durch das Trainieren Ihrer Einbettungen kann der Hyperparameter **d_model** auf jeden gewünschten Wert eingestellt werden. Wenn Sie die Einbettungen übertragen, müssen Sie den Hyperparameter **d_model** auf denselben Wert wie die übertragenen Einbettungen setzen.

Der Hyperparameter **dff** gibt die Größe der dichten Feedforward-Schichten an. Der Hyperparameter **num_heads** legt die Anzahl der Köpfe der Aufmerksamkeitsschichten fest. Schließlich gibt dropout_rate einen Dropout-Prozentsatz an, um Überanpassungen vorzubeugen. Dropout haben wir bereits früher in diesem Buch besprochen.

## Im Inneren eines Transformators

In diesem Abschnitt untersuchen wir das Innenleben eines Transformators, damit Sie sich mit wesentlichen Konzepten vertraut machen, wie z. B.:

* Einbettungen
* Positionskodierung
* Aufmerksamkeit und Selbstaufmerksamkeit
* Restverbindung

In Abbildung 10.TRANS-2 sehen Sie ein Diagramm eines Transformators auf niedrigerer Ebene.

**Abbildung 10.TRANS-2: Architekturdiagramm aus dem Dokument**
![Attention is All you Need](https://data.heatonresearch.com/images/jupyter/transformer-2.jpg)

Während das ursprüngliche Transformer-Papier den Titel „Aufmerksamkeit ist alles, was Sie brauchen“ trägt, ist Aufmerksamkeit nicht der einzige Schichttyp, den Sie benötigen. Der Transformer enthält auch dichte Schichten. Der Titel „Aufmerksamkeit und dichte Schichten sind alles, was Sie brauchen“ ist jedoch nicht so eingängig.

Der Transformator beginnt mit der Tokenisierung des eingegebenen englischen Satzes. Token können Wörter sein, müssen es aber nicht. Im Allgemeinen werden bekannte Wortteile tokenisiert und werden zu Bausteinen längerer Wörter. Diese Tokenisierung ermöglicht es, gängige Suffixe und Präfixe unabhängig von ihrem Stammwort zu verstehen. Jedes Token wird zu einem numerischen Index, den der Transformator zum Nachschlagen des Vektors verwendet. Es gibt mehrere spezielle Token:

* Index 0 = Pad
* Index 1 = Unbekannt
* Index 2 = Starttoken
* Index 3 = Endtoken

Der Transformator verwendet Index 0, wenn wir ungenutzten Platz am Ende eines Tensors auffüllen müssen. Index 1 ist für unbekannte Wörter. Die Start- und Endtoken werden durch die Indizes 2 und 3 bereitgestellt.

Die Token-Vektoren sind einfach die Eingaben für die Aufmerksamkeitsebenen; es gibt keine implizite Reihenfolge oder Position. Der Transformator fügt den Token-Vektoren die Steigungen einer Sinus- und Cosinuswelle hinzu, um die Position zu kodieren.

Aufmerksamkeitsebenen haben drei Eingaben: Schlüssel (k), Wert (v) und Abfrage (q). Diese Ebene ist selbstaufmerksam, wenn Abfrage, Schlüssel und Wert gleich sind. Die Schlüssel- und Wertepaare geben die Informationen an, auf die die Abfrage angewendet wird. Die Aufmerksamkeitsebene lernt, auf welche Datenpositionen sie sich konzentrieren soll.

Der Transformator präsentiert die positionscodierten Einbettungsvektoren dem ersten Self-Attention-Segment in der coderschicht. Die Ausgabe der Aufmerksamkeit wird normalisiert und wird schließlich zum verborgenen Zustand, nachdem alle coderschichten verarbeitet wurden.

Der verborgene Zustand wird nur einmal pro Abfrage berechnet. Sobald der eingegebene spanische Satz einen verborgenen Zustand aufweist, wird dieser Wert dem Decoder wiederholt angezeigt, bis der Decoder den endgültigen spanischen Satz bildet.

Dieser Abschnitt bot eine allgemeine Einführung in Transformatoren. Im nächsten Teil werden wir den coder implementieren und auf Zeitreihen anwenden. Im folgenden Kapitel werden wir [Hugging Face](https://huggingface.co/)-Transformatoren verwenden, um natürliche Sprachverarbeitung durchzuführen.




# Modul 6 Aufgabe

Die fünfte Aufgabe finden Sie hier: [assignment 6](https://github.com/jeffheaton/app_deep_learning/blob/main/assignments/assignment_yourname_t81_558_class6.ipynb)
